# OptBench Progress Monitor

Tracks progress of the per-(exp, model, seed) slurm jobs launched by `check_slurm.sh`.

- **Section 1** — done/total per (exp, model, seed), pulling recipe counts directly from `exp1.py`/`exp2.py` and finished runs from wandb.
- **Section 2** — hours-remaining estimates per slurm job, from prior finished runtimes on the same model.

In [ ]:
import importlib.util
import re
from pathlib import Path

import pandas as pd
import wandb

# scripts/opt-bench has a '-' in its name so it is not a valid python package;
# load exp1/exp2 by file path instead.
_HERE = Path.cwd() if Path.cwd().name == 'opt-bench' else Path('scripts/opt-bench')

def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

exp1 = _load('exp1', _HERE / 'exp1.py')
exp2 = _load('exp2', _HERE / 'exp2.py')

print(f'exp1 whitebox recipes: {len(exp1.WHITEBOX_OPTIMIZER_CONFIGS)} + 1 nanogcg')
print(f'exp2 variants: {len(exp2._build_variants(n_layers=0))}')
print(f'msg_ids: {len(exp1.MSG_IDS)}   seeds: {exp1.SEEDS}')

In [ ]:
# ─── Job matrix (must match check_slurm.sh / run_all.sh) ──────────────────
LETTER_TO_MODEL = {
    'l': 'meta-llama/Llama-3.1-8B-Instruct',
    'g': 'google/gemma-3-12b-it',
    'q': 'Qwen/Qwen3-8B',
    'm': 'google/gemma-4-26B-A4B-it',
}
# exp2 is pinned to gemma-3-12b-it (see run_all.sh EXP2_MODELS).
EXP1_LETTERS = list(LETTER_TO_MODEL)  # l, g, q, m
EXP2_LETTERS = ['g']
# exp2 has two sub-runs per slurm job: '2_single' (per msg_id × seed) and
# '2_multi' (one universal trigger per seed across all msgs). Tracked separately.
EXP_LETTERS = {'1': EXP1_LETTERS, '2_single': EXP2_LETTERS, '2_multi': EXP2_LETTERS}
MODEL_SHORTS = {letter: m.split('/')[-1] for letter, m in LETTER_TO_MODEL.items()}
SEEDS = list(exp1.SEEDS)
MSG_IDS = list(exp1.MSG_IDS)

WANDB_ENTITY = exp1.WANDB_ENTITY
EXP1_PROJECT = exp1.WANDB_PROJECT           # tropt-optbench
EXP1_BB_PROJECT = exp1.WANDB_PROJECT_BB     # tropt-optbench-bb (OpenAI blackbox)
EXP2_PROJECT = exp2.WANDB_PROJECT           # tropt-enhancebench

BLACKBOX_MODEL = 'openai/gpt-5-nano'
BLACKBOX_MODEL_SHORT = BLACKBOX_MODEL.split('/')[-1]

# Recipe names per experiment / sub-experiment
EXP1_RECIPES = [c.name for c in exp1.WHITEBOX_OPTIMIZER_CONFIGS] + ['nanogcg']
EXP1_BB_RECIPES = [c.name for c in exp1.BLACKBOX_OPTIMIZER_CONFIGS]
EXP2_RECIPES = [v.name for v in exp2._build_variants(n_layers=0)]
# `multi` no longer skips any variants — runs all of them (matches exp2.py).
EXP2_MULTI_RECIPES = list(EXP2_RECIPES)

# Expected runs per slurm job
def expected_count(exp_kind: str) -> int:
    if exp_kind == '1':
        return len(EXP1_RECIPES) * len(MSG_IDS)
    if exp_kind == '1bb':
        # exp1chat is a single job covering ALL seeds × ALL msgs × blackbox optimizers
        return len(EXP1_BB_RECIPES) * len(MSG_IDS) * len(SEEDS)
    if exp_kind == '2_single':
        return len(EXP2_RECIPES) * len(MSG_IDS)
    if exp_kind == '2_multi':
        # one universal trigger per recipe; seed is fixed within a slurm job
        return len(EXP2_MULTI_RECIPES)
    raise ValueError(exp_kind)

print(f'Expected runs per exp1 job:           {expected_count("1")}')
print(f'Expected runs per exp1chat:           {expected_count("1bb")}')
print(f'Expected runs per exp2 job (single):  {expected_count("2_single")}')
print(f'Expected runs per exp2 job (multi):   {expected_count("2_multi")}')


In [ ]:
# ─── Fetch finished runs from wandb ──────────────────────────────────────
api = wandb.Api()

def fetch_runs(project: str, state: str | None = None):
    filters = {'state': state} if state else None
    try:
        return list(api.runs(f'{WANDB_ENTITY}/{project}', filters=filters))
    except Exception as e:
        print(f'  [warn] {project}: {e}')
        return []

exp1_runs = fetch_runs(EXP1_PROJECT, state='finished')
exp1_bb_runs = fetch_runs(EXP1_BB_PROJECT, state='finished')
exp2_runs = fetch_runs(EXP2_PROJECT, state='finished')
print(f'finished runs — exp1: {len(exp1_runs)}  exp1_bb: {len(exp1_bb_runs)}  exp2: {len(exp2_runs)}')

# Patterns for parsing the run_name fields.
# exp1 whitebox/nanogcg AND exp1 blackbox share the same prefix 'optbench['.
RE_EXP1 = re.compile(r'^optbench\[(?P<recipe>[^,]+),(?P<model>[^,]+),m=(?P<msg>\d+),s=(?P<seed>\d+)\]$')
RE_EXP2_SINGLE = re.compile(r'^enhancebench\[(?P<recipe>[^,]+),(?P<model>[^,]+),m=(?P<msg>\d+),s=(?P<seed>\d+)\]$')
RE_EXP2_MULTI = re.compile(r'^enhancebench_multi\[(?P<recipe>[^,]+),(?P<model>[^,]+),s=(?P<seed>\d+)\]$')

# (exp_value, regex). exp_value '1' is later split into '1' / '1bb' by source project.
_PARSERS = (
    ('1',        RE_EXP1),
    ('2_multi',  RE_EXP2_MULTI),
    ('2_single', RE_EXP2_SINGLE),
)

def parse_run(r, source_project: str):
    for exp_value, regex in _PARSERS:
        m = regex.match(r.name or '')
        if m:
            d = m.groupdict()
            d['run'] = r
            if exp_value == '1':
                d['exp'] = '1bb' if source_project == EXP1_BB_PROJECT else '1'
            else:
                d['exp'] = exp_value
            return d
    return None

parsed = []
for r in exp1_runs:
    p = parse_run(r, EXP1_PROJECT)
    if p: parsed.append(p)
for r in exp1_bb_runs:
    p = parse_run(r, EXP1_BB_PROJECT)
    if p: parsed.append(p)
for r in exp2_runs:
    p = parse_run(r, EXP2_PROJECT)
    if p: parsed.append(p)
print(f'parsed: {len(parsed)}')
print(f'  exp=1:        {sum(1 for p in parsed if p["exp"] == "1")}')
print(f'  exp=1bb:      {sum(1 for p in parsed if p["exp"] == "1bb")}')
print(f'  exp=2_single: {sum(1 for p in parsed if p["exp"] == "2_single")}')
print(f'  exp=2_multi:  {sum(1 for p in parsed if p["exp"] == "2_multi")}')


In [ ]:
# ─── Progress table: done / total per slurm job ──────────────────────────
# exp2 is split into two rows per (model, seed): one for the per-msg single
# attacks and one for the universal-trigger multi attack — they have very
# different totals and failure modes, so a single combined number was hiding
# whether multi runs were missing or just lots of single-msg gaps.
rows = []

for exp in ['1', '2_single', '2_multi']:
    total = expected_count(exp)
    for letter in EXP_LETTERS[exp]:
        short = MODEL_SHORTS[letter]
        for seed_idx, seed in enumerate(SEEDS, start=1):
            done = sum(
                1 for p in parsed
                if p['exp'] == exp and p['model'] == short and int(p['seed']) == seed
            )
            # Slurm job name uses the bare exp digit — single & multi run inside the same job.
            slurm_exp = exp.split('_', 1)[0]  # '1' | '2'
            rows.append({
                'job': f'exp{slurm_exp}{letter}{seed_idx}',
                'exp': exp,
                'model': short,
                'seed': seed,
                'done': done,
                'total': total,
                'pct': f'{100*done/total:5.1f}%' if total else '   - ',
            })

# exp1chat: the single OpenAI blackbox job — aggregates over all seeds and msgs.
bb_total = expected_count('1bb')
bb_done = sum(
    1 for p in parsed
    if p['exp'] == '1bb' and p['model'] == BLACKBOX_MODEL_SHORT
)
rows.append({
    'job': 'exp1chat',
    'exp': '1bb',
    'model': BLACKBOX_MODEL_SHORT,
    'seed': 'all',
    'done': bb_done,
    'total': bb_total,
    'pct': f'{100*bb_done/bb_total:5.1f}%',
})

progress_df = pd.DataFrame(rows)
progress_df


## Section 2 — Remaining-hours estimate

For each recipe on a given model, take the runtime of the most recent finished run as a proxy for the cost of any future run of that recipe on that model (just a rough estimate — no averaging). Then, for each slurm job, sum the runtimes of recipes that have *not* yet finished for that (model, seed) to get the hours remaining.

In [ ]:
# ─── Build per-recipe runtime lookup from finished runs ──────────────────
# runtime_by_recipe[(exp, model, recipe)] = median runtime (seconds) of a single wandb run
# for that (exp, model, recipe) — i.e. ONE (msg, seed) sample, not the full sweep.
# Note: '2_single' and '2_multi' are kept as separate keys, since multi runs can
# be much longer (more templates per loss step + larger FLOP budget).
from statistics import median

_rt_buckets: dict[tuple[str, str, str], list[float]] = {}
for p in parsed:
    r = p['run']
    rt = r.summary.get('_runtime') or getattr(r, '_attrs', {}).get('runtime')
    if rt is None:
        continue
    _rt_buckets.setdefault((p['exp'], p['model'], p['recipe']), []).append(float(rt))

runtime_by_recipe = {k: median(v) for k, v in _rt_buckets.items()}

# Fallback: average runtime per (exp, recipe) across any model — used when the
# target model hasn't finished even one run of that recipe yet.
_any_buckets: dict[tuple[str, str], list[float]] = {}
for (exp, _model, recipe), rts in _rt_buckets.items():
    _any_buckets.setdefault((exp, recipe), []).extend(rts)
runtime_by_recipe_any_model = {k: median(v) for k, v in _any_buckets.items()}

print(f'per-(exp,model,recipe) entries: {len(runtime_by_recipe)}')
print(f'per-(exp,recipe) fallback:      {len(runtime_by_recipe_any_model)}')


def lookup_runtime(exp: str, model: str, recipe: str) -> tuple[float | None, str]:
    """Return (seconds, source) where source ∈ {'same_model', 'any_model', 'missing'}."""
    v = runtime_by_recipe.get((exp, model, recipe))
    if v is not None:
        return v, 'same_model'
    v = runtime_by_recipe_any_model.get((exp, recipe))
    if v is not None:
        return v, 'any_model'
    return None, 'missing'


def recipes_for(exp: str) -> list[str]:
    return {
        '1':        EXP1_RECIPES,
        '1bb':      EXP1_BB_RECIPES,
        '2_single': EXP2_RECIPES,
        '2_multi':  EXP2_MULTI_RECIPES,
    }[exp]


def expected_per_recipe(exp: str) -> int:
    # How many wandb runs a *single* recipe contributes to one slurm job (seed fixed).
    if exp == '1':
        return len(MSG_IDS)
    if exp == '1bb':
        return len(MSG_IDS) * len(SEEDS)
    if exp == '2_single':
        return len(MSG_IDS)
    if exp == '2_multi':
        return 1
    raise ValueError(exp)


def done_by_recipe(exp: str, model: str, seed) -> dict[str, int]:
    """Count finished runs per recipe for this (exp, model, seed)."""
    counts: dict[str, int] = {}
    for p in parsed:
        if p['exp'] != exp or p['model'] != model:
            continue
        if seed != 'all' and int(p['seed']) != seed:
            continue
        counts[p['recipe']] = counts.get(p['recipe'], 0) + 1
    return counts


def estimate_remaining_hours(exp: str, model: str, seed) -> tuple[float, int, int]:
    """(hours, n_recipes_using_fallback, n_recipes_no_rt_data)."""
    done = done_by_recipe(exp, model, seed)
    per_recipe_budget = expected_per_recipe(exp)
    total_s = 0.0
    fallback_n = 0
    missing_n = 0
    for recipe in recipes_for(exp):
        remaining = max(0, per_recipe_budget - done.get(recipe, 0))
        if remaining == 0:
            continue
        rt, src = lookup_runtime(exp, model, recipe)
        if rt is None:
            missing_n += 1
            continue
        if src == 'any_model':
            fallback_n += 1
        total_s += rt * remaining
    return total_s / 3600.0, fallback_n, missing_n


est_rows = []
for exp in ['1', '2_single', '2_multi']:
    for letter in EXP_LETTERS[exp]:
        short = MODEL_SHORTS[letter]
        for seed_idx, seed in enumerate(SEEDS, start=1):
            hrs, fb, miss = estimate_remaining_hours(exp, short, seed)
            slurm_exp = exp.split('_', 1)[0]
            est_rows.append({
                'job': f'exp{slurm_exp}{letter}{seed_idx}',
                'exp': exp,
                'est_hours_left': round(hrs, 2),
                'recipes_fallback_rt': fb,
                'recipes_no_rt': miss,
            })
hrs, fb, miss = estimate_remaining_hours('1bb', BLACKBOX_MODEL_SHORT, 'all')
est_rows.append({
    'job': 'exp1chat',
    'exp': '1bb',
    'est_hours_left': round(hrs, 2),
    'recipes_fallback_rt': fb,
    'recipes_no_rt': miss,
})
estimate_df = pd.DataFrame(est_rows)
estimate_df


In [ ]:
# ─── Combined view ───────────────────────────────────────────────────────
# Merge on (job, exp): exp2 contributes two rows per slurm job (single + multi)
# with the same `job` value, so merging on job alone would cross-join.
combined = progress_df.merge(
    estimate_df[['job', 'exp', 'est_hours_left', 'recipes_fallback_rt', 'recipes_no_rt']],
    on=['job', 'exp'],
)
combined.sort_values(['exp', 'job']).reset_index(drop=True)


## Section 3 — Per-(recipe, model, msg) run counts

Counts finished runs for each `(recipe, model, msg)` triple. With 3 seeds we expect exactly **3** runs per cell; anything less flags missing seeds.

Rendered as a heatmap per experiment: green = complete, yellow = partial, red = missing. For exp2's `multi` runs (no `msg` axis, one trigger per seed) we instead show seed coverage per `(model, recipe)` — expected = 1 per cell.

In [ ]:
# ─── Section 3: per-(recipe, model, msg) run counts as heatmaps ─────────
# Expected = len(SEEDS) per (recipe, model, msg) cell for *_single experiments.
# For '2_multi' there is no msg axis, so we plot (model, recipe) × seed instead.
import matplotlib.pyplot as plt
import seaborn as sns


def per_recipe_model_msg_counts(exp: str, models_shorts: list[str]) -> pd.DataFrame:
    recipes = recipes_for(exp)
    rows = []
    for short in models_shorts:
        for recipe in recipes:
            for msg in MSG_IDS:
                n = sum(
                    1 for p in parsed
                    if p['exp'] == exp
                    and p['model'] == short
                    and p['recipe'] == recipe
                    and 'msg' in p
                    and int(p['msg']) == int(msg)
                )
                rows.append({'model': short, 'recipe': recipe, 'msg': int(msg), 'count': n})
    df = pd.DataFrame(rows)
    pivot = df.pivot_table(index=['model', 'recipe'], columns='msg', values='count', fill_value=0)
    ordered_idx = [(m, r) for m in models_shorts for r in recipes if (m, r) in pivot.index]
    return pivot.reindex(ordered_idx)


def per_recipe_model_seed_counts(exp: str, models_shorts: list[str]) -> pd.DataFrame:
    """For multi runs: count finished runs per (model, recipe, seed). Expected = 1 per cell."""
    recipes = recipes_for(exp)
    rows = []
    for short in models_shorts:
        for recipe in recipes:
            for seed in SEEDS:
                n = sum(
                    1 for p in parsed
                    if p['exp'] == exp
                    and p['model'] == short
                    and p['recipe'] == recipe
                    and int(p['seed']) == seed
                )
                rows.append({'model': short, 'recipe': recipe, 'seed': seed, 'count': n})
    df = pd.DataFrame(rows)
    pivot = df.pivot_table(index=['model', 'recipe'], columns='seed', values='count', fill_value=0)
    ordered_idx = [(m, r) for m in models_shorts for r in recipes if (m, r) in pivot.index]
    return pivot.reindex(ordered_idx)


# (exp, models, builder, expected_per_cell, x_label)
_LAYOUTS = [
    ('1',        [MODEL_SHORTS[l] for l in EXP1_LETTERS], per_recipe_model_msg_counts,  len(SEEDS), 'msg_id'),
    ('2_single', [MODEL_SHORTS[l] for l in EXP2_LETTERS], per_recipe_model_msg_counts,  len(SEEDS), 'msg_id'),
    ('2_multi',  [MODEL_SHORTS[l] for l in EXP2_LETTERS], per_recipe_model_seed_counts, 1,          'seed'),
    ('1bb',      [BLACKBOX_MODEL_SHORT],                  per_recipe_model_msg_counts,  len(SEEDS), 'msg_id'),
]

for exp, models, builder, expected_cell, xlabel in _LAYOUTS:
    pivot = builder(exp, models)
    if pivot.empty:
        print(f'exp{exp}: (no runs yet)')
        continue
    labels = [f'{m} · {r}' for m, r in pivot.index]
    fig_h = max(1.2, 0.32 * len(pivot) + 0.8)
    fig, ax = plt.subplots(figsize=(8, fig_h))
    sns.heatmap(
        pivot.values,
        ax=ax,
        vmin=0, vmax=expected_cell,
        cmap='RdYlGn',
        annot=True, fmt='.0f',
        cbar_kws={'label': f'count (of {expected_cell})', 'ticks': list(range(expected_cell + 1))},
        linewidths=0.5, linecolor='white',
        xticklabels=[str(c) for c in pivot.columns],
        yticklabels=labels,
    )
    # Horizontal separators between model groups.
    model_col = [m for m, _ in pivot.index]
    for i in range(1, len(model_col)):
        if model_col[i] != model_col[i - 1]:
            ax.axhline(i, color='black', linewidth=1.2)
    n_complete = int((pivot.min(axis=1) >= expected_cell).sum())
    ax.set_title(f'exp{exp} — coverage per (model, recipe, {xlabel})  '
                 f'[{n_complete}/{len(pivot)} rows complete]')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
